# Milestone 3: Model Improvement

**Deadline:** `Mar 1, 2026`

1. Dimensionality Reduction
2. Feature Selection
3. Retrain the models
    - [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
    - [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
    - [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)

## Import libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

## Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

## Question 1

Split the train dataset using train_test_split with random_state=42 such that 80% is training data and remaining 20% is validation data. store the data as X_train, X_val, y_train, y_val. Lets say training data (X_train) has shape (a,b) where a is number of rows and b is the number of features, similarly validation set (X_val) has shape (c,d) where c is the number of rows and d is the number of features. what will be the value of a + c ?

In [4]:
X = train.drop(columns=["post_id", "label"])
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train.shape[0] + X_val.shape[0]

198000

## Question 2

After converting created_date to a datetime object and extracting the day, month, and year for your X_train and X_val created in the first question, identify the most frequently occurring month across all years in X_train. Which month is it?

In [6]:
X_train["created_date"] = pd.to_datetime(X_train["created_date"])
X_val["created_date"] = pd.to_datetime(X_val["created_date"])

X_train["day"] = X_train["created_date"].dt.day
X_train["month"] = X_train["created_date"].dt.month
X_train["year"] = X_train["created_date"].dt.year

X_val["day"] = X_val["created_date"].dt.day
X_val["month"] = X_val["created_date"].dt.month
X_val["year"] = X_val["created_date"].dt.year

X_train["month"].mode()[0]

np.int32(5)

## Question 3

Impute the null values of categorical features with the value 'none'. Encode only `'religion', 'gender', 'race'` features using one hot encoding, setting `handle_unknown='ignore'`. Make sure the output is a pandas dataframe. Let's say the shape of X_train after imputing is (a,b), what is the value of b?\
**Note:** make sure to transform X_val as well using one hot encoding.

In [7]:
cat_features = X_train.select_dtypes(include="str").columns
cat_features

Index(['race', 'religion', 'gender', 'comment'], dtype='str')

In [8]:
X_train[cat_features] = X_train[cat_features].fillna("none")
X_val[cat_features] = X_val[cat_features].fillna("none")

In [9]:
one_hot = ColumnTransformer(
    [
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["religion", "gender", "race"],
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train = one_hot.fit_transform(X_train)
X_val = one_hot.transform(X_val)

In [10]:
X_train.shape[1]

32

## Question 4

Apply **CountVectorizer** to the column **comment** of X_train and X_val obtained in previous questions, what is the sum of the counts for the document at index 1 (the second row) of the transformed X_train sparse matrix?\
**Note:** Please ensure to transform X_val as well.

In [11]:
count_vec = CountVectorizer()

X_train_count = count_vec.fit_transform(X_train["comment"])
X_val_count = count_vec.transform(X_val["comment"])

In [12]:
X_train_count[1].sum()

np.int64(41)

## Question 5

Convert **'disability'** feature into integer type, with True being mapped to 1 and False mapped to 0. What is the sum of all the 'disability values in X_train and X_val after the transformation?

In [13]:
X_train["disability"] = X_train["disability"].map({True: 1, False: 0})
X_val["disability"] = X_val["disability"].map({True: 1, False: 0})

In [14]:
X_train["disability"].sum() + X_val["disability"].sum()

np.int64(2743)

## Question 6

Scale the numeric features using **StandardScaler**. What are the number of features seen during fit for X_train? Ensure that you drop all **datetime** columns first.

In [15]:
dt_cols = X_train.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns
X_train_no_dt = X_train.drop(columns=dt_cols)

num_features = X_train_no_dt.select_dtypes(include="number").columns
scaler = StandardScaler()
scaler.fit(X_train_no_dt[num_features])

scaler.n_features_in_  # type: ignore

30

## Question 7

Preprocess the data in the following manner and train the model:

Use the full raw data and first split it into training and validation sets so that 80% of the data forms **X_train, y_train** and the remaining 20% forms **X_val, y_val**.

**After splitting:**

1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Transform the "comment" feature using **TfidfVectorizer** with stop_words='english'. Make sure to transform X_val accordingly.
7. Encode all the categorical features using one hot encoder with handle_unknown='ignore'. Make sure to transform X_val accordingly.
8. Now train a Multinomial Naive Bayes model on the preprocessed data. Use a pipeline if you need to.

What is the macro F1 score obtained on the train dataset?

**Note:** You may train Multinomial Naive Bayes model on the preprocessed data which has been obtained from previous questions and give the answer, both ways the answers will be accepted.

If some feature is creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [16]:
def preprocess(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [17]:
X_train_proc = preprocess(X_train)
X_val_proc = preprocess(X_val)

In [18]:
preprocessor = ColumnTransformer(
    transformers=[("tfidf", TfidfVectorizer(stop_words="english"), "comment")],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [19]:
y_train_pred = pipeline.predict(X_train_proc)
f1_score(y_train, y_train_pred, average="macro")

0.4841311504632059

## Question 8

What is the macro F1 score obtained on the validation dataset?

In [20]:
y_val_pred = pipeline.predict(X_val_proc)
f1_score(y_val, y_val_pred, average="macro")

0.45926776710921624

## Question 9

Process the data in the following manner as given below and train the model.

Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms **X_val, y_val**.

After splitting:
1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Create a new binary categorical column named **is_weekend** where Saturday and Sunday are represented as **1** and all other days as **0**.
7. Apply **TfidfVectorizer(stop_words='english')** to the comment column. Make sure to transform X_val accordingly.
8. Apply **OneHotEncoder(handle_unknown='ignore')** to the categorical features including newly created categorical feature **is_weekend**. Make sure to transform X_val accordingly.

After all above transformation, train **MultinomialNB** model.

After training, use the model to predict the labels for X_train and calculate the **Macro F1 score**. What is the resulting score rounded to four decimal places?

**Note:** If some features are creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [21]:
def preprocess_v2(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df["is_weekend"] = df["created_date"].dt.dayofweek.apply(
        lambda x: 1 if x >= 5 else 0
    )
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [22]:
X_train_proc_v2 = preprocess_v2(X_train)
X_val_proc_v2 = preprocess_v2(X_val)

In [23]:
preprocessor = ColumnTransformer(
    [
        ("tfidf", TfidfVectorizer(stop_words="english"), "comment"),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["is_weekend"]),
    ],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc_v2, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...), ('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [24]:
y_train_pred = pipeline.predict(X_train_proc_v2)
f1_score(y_train, y_train_pred, average="macro")

0.4840696062885761

## Question 10

What is the macro F1-score on validation set for the previous question's setup?

In [25]:
y_val_pred = pipeline.predict(X_val_proc_v2)
f1_score(y_val, y_val_pred, average="macro")

0.4592333361559341